# 🎙 Training — AI Audiobook Reader (RVC) — Local

Trains your personal **Applio RVC** voice model from `voice_training.wav`.

- **GPU**: NVIDIA RTX 3090 (CUDA 12)
- **Duration**: ~15–30 minutes at 150 epochs
- **Run once**: model saved to `rvc-training-inference/models/`

**Before running:** select the `vclone_venv` Jupyter kernel.
If it's missing, run Cell 1 first with any kernel, then:
```
vclone_venv/bin/python -m ipykernel install --user --name vclone_venv --display-name "vclone_venv"
```
Then restart the kernel using `vclone_venv`.


In [ ]:
# ── Cell 1: Install — clone Applio and install all deps into vclone_venv ──────
# Safe to re-run; uv skips already-installed packages.
import sys, os, subprocess, shutil

BASE_DIR   = '/home/zero/Desktop/Explore/VoiceCloning/rvc-training-inference'
APPLIO_DIR = f'{BASE_DIR}/Applio'
VENV_PY    = '/home/zero/Desktop/Explore/VoiceCloning/vclone_venv/bin/python'

print(f'Python (kernel):  {sys.executable}')
print(f'Python (venv):    {VENV_PY}')
print(f'Base dir: {BASE_DIR}')

# Clone Applio v3.6.2
if not os.path.exists(APPLIO_DIR):
    print('Cloning Applio v3.6.2 ...')
    subprocess.run(['git', 'clone', 'https://github.com/IAHispano/Applio',
                    '--branch', '3.6.2', '--single-branch', APPLIO_DIR],
                   check=True)
    print('✓ Applio cloned')
else:
    print('✓ Applio already present')

# System lib needed by pyaudio (part of Applio deps)
r_apt = subprocess.run(
    ['sudo', 'apt-get', 'install', '-y', '-q', 'portaudio19-dev'],
    capture_output=True, text=True)
if r_apt.returncode == 0:
    print('✓ portaudio19-dev installed')
else:
    print('portaudio note:', r_apt.stderr[-200:].strip() or r_apt.stdout[-200:].strip())

# Install ipykernel so vclone_venv can be selected as a Jupyter kernel
uv_bin = shutil.which('uv') or '/home/zero/.local/bin/uv'
subprocess.run([uv_bin, 'pip', 'install', '--python', VENV_PY, '-q', 'ipykernel'],
               capture_output=True)

# Install Applio runtime deps into vclone_venv
print(f'\nInstalling Applio deps via uv → {VENV_PY} ...')
print('(~5 min first time; cached on subsequent runs)\n')
r_uv = subprocess.run([
    uv_bin, 'pip', 'install', '--python', VENV_PY, '-q',
    '-r', f'{APPLIO_DIR}/requirements.txt',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu124',
    '--index-strategy', 'unsafe-best-match',
], text=True, capture_output=True)
if r_uv.returncode != 0:
    print(r_uv.stderr[-3000:])
    raise RuntimeError('uv pip install failed — see output above')
print('✓ Applio dependencies installed')

# Extra packages used by this notebook
r_extra = subprocess.run([
    uv_bin, 'pip', 'install', '--python', VENV_PY, '-q',
    'pydub', 'matplotlib', 'tensorboard', 'scipy',
], text=True, capture_output=True)
if r_extra.returncode != 0:
    print(r_extra.stderr[-500:])
print('✓ Extra packages (pydub, matplotlib, tensorboard) installed')

# Download Applio pretrained models (rmvpe, contentvec, hifigan)
print('\nDownloading pretrained models ...')
r_pre = subprocess.run(
    [VENV_PY, 'core.py', 'prerequisites',
     '--models', 'True', '--pretraineds_hifigan', 'True'],
    capture_output=True, text=True, cwd=APPLIO_DIR
)
if r_pre.returncode != 0:
    print('prerequisites stdout:', r_pre.stdout[-500:])
    print('prerequisites stderr:', r_pre.stderr[-500:])
else:
    print('✓ Pretrained models ready')

print('\n✅ Cell 1 complete. If you changed kernels, restart and select vclone_venv.')


In [ ]:
# ── Cell 2: Config ─────────────────────────────────────────────────────────────
import sys, os, torch

BASE_DIR   = '/home/zero/Desktop/Explore/VoiceCloning/rvc-training-inference'
APPLIO_DIR = f'{BASE_DIR}/Applio'
MODEL_NAME  = 'user_voice'
SAMPLE_RATE = 40000   # Standard RVC sample rate
EPOCHS      = 150     # Increase to 300–500 for higher quality (takes longer)

if torch.cuda.is_available():
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**2} MB')
else:
    print('⚠  No CUDA GPU detected — training will be extremely slow on CPU')
    print('   Set EPOCHS low (e.g. 10) to test, or switch to a GPU machine')

# Ensure output dirs exist
os.makedirs(f'{BASE_DIR}/models', exist_ok=True)
os.makedirs(f'{BASE_DIR}/dataset/{MODEL_NAME}', exist_ok=True)

print(f'✓ Python:     {sys.executable}')
print(f'✓ Base dir:   {BASE_DIR}')
print(f'✓ Model name: {MODEL_NAME}')
print(f'✓ Epochs:     {EPOCHS}')


In [ ]:
# ── Cell 3: Prepare audio ──────────────────────────────────────────────────────
import os, subprocess

src_wav     = f'{BASE_DIR}/voice_training.wav'
dataset_dir = f'{BASE_DIR}/dataset/{MODEL_NAME}'
output_wav  = f'{dataset_dir}/voice_recording.wav'

assert os.path.exists(src_wav), f'voice_training.wav not found at {src_wav}'
print(f'✓ Source: {src_wav} ({os.path.getsize(src_wav)//1024} KB)')

os.makedirs(dataset_dir, exist_ok=True)

# Normalise loudness, resample to 40 kHz mono
r = subprocess.run([
    'ffmpeg', '-y', '-i', src_wav,
    '-ar', str(SAMPLE_RATE), '-ac', '1',
    '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11',
    output_wav
], capture_output=True, text=True)
if r.returncode != 0:
    print('FFmpeg error:', r.stderr[-500:])
    raise RuntimeError('Audio conversion failed')

duration = float(subprocess.check_output([
    'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
    '-of', 'default=noprint_wrappers=1:nokey=1', output_wav
]).decode().strip())

print(f'✓ Audio ready: {duration:.1f}s ({duration/60:.1f} min) at {SAMPLE_RATE} Hz')
print(f'  Output: {output_wav}')
if duration < 120:
    print(f'⚠  Only {duration:.0f}s — 5+ minutes gives much better quality.')
else:
    print(f'✓ Duration looks good.')


In [ ]:
# ── Cell 4: Preprocess → Extract → Index ──────────────────────────────────────
import sys, os, glob, subprocess

VENV_PY    = sys.executable
LOG_DIR    = f'{APPLIO_DIR}/logs/{MODEL_NAME}'
dataset_dir = f'{BASE_DIR}/dataset/{MODEL_NAME}'

def dump_log_tree(label):
    entries = sorted(glob.glob(f'{LOG_DIR}/**/*', recursive=True))
    lines = [f'  {os.path.relpath(p, LOG_DIR)}  ({os.path.getsize(p):,} B)'
             for p in entries if os.path.isfile(p)]
    print(f'\n[diag] {label} — {len(lines)} files in logs/{MODEL_NAME}:')
    for l in lines[:20]: print(l)

def run_step(cmd, label):
    print(f'\n{"-"*55}\n{label} ...')
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=APPLIO_DIR)
    out = r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout
    if out.strip(): print(out)
    if r.stderr.strip(): print('STDERR:', r.stderr[-500:])
    if r.returncode != 0:
        dump_log_tree('after failure')
        raise RuntimeError(f'{label} failed (exit {r.returncode})')
    print(f'✓ {label} complete')

run_step([
    VENV_PY, 'core.py', 'preprocess',
    '--model_name', MODEL_NAME,
    '--dataset_path', dataset_dir,
    '--sample_rate', str(SAMPLE_RATE),
    '--cut_preprocess', 'Automatic',
], 'Preprocessing audio')

run_step([
    VENV_PY, 'core.py', 'extract',
    '--model_name', MODEL_NAME,
    '--f0_method', 'rmvpe',
    '--sample_rate', str(SAMPLE_RATE),
    '--embedder_model', 'contentvec',
    '--include_mutes', '0',
    '--cpu_cores', '4',
], 'Extracting F0 + embeddings')

run_step([
    VENV_PY, 'core.py', 'index',
    '--model_name', MODEL_NAME,
    '--index_algorithm', 'Auto',
], 'Building FAISS index')

print('\n✅ All preprocessing stages done. Run Cell 5 to start training.')
dump_log_tree('after pipeline')


In [ ]:
# ── Cell 5: Train with live monitoring ────────────────────────────────────────
# Training runs in a subprocess. The main thread polls stdout + TensorBoard events
# every 30 s and redraws a 4-panel live chart in-place.
#
# Panels:
#   1. Gen & Disc loss (smoothed, from stdout)  — primary convergence signal
#   2. Mel spectrogram loss (from TensorBoard)  — best perceptual quality proxy
#   3. Sub-losses: feat-match, KL, adversarial  — GAN component health
#   4. Learning rate (log scale)                — decay schedule verification
import sys, os, re, subprocess, threading, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

VENV_PY      = sys.executable
LOG_DIR      = f'{APPLIO_DIR}/logs/{MODEL_NAME}'
REFRESH_SECS = 30   # seconds between plot redraws

# ── stdout parser ─────────────────────────────────────────────────────────────
# Applio prints per-epoch records like:
#   user_voice | epoch=5 | step=250 | ... | lowest_value=0.456 (epoch 3 ...) |
#   smoothed_loss_gen=0.456 | smoothed_loss_disc=0.789
RE_EPOCH  = re.compile(r'epoch=(\d+)')
RE_STEP   = re.compile(r'step=(\d+)')
RE_S_GEN  = re.compile(r'smoothed_loss_gen=([\d.]+)')
RE_S_DISC = re.compile(r'smoothed_loss_disc=([\d.]+)')
RE_BEST   = re.compile(r'lowest_value=([\d.]+)')

epoch_records = []   # [{epoch, step, s_gen, s_disc, best}]  — one per epoch
log_lines     = []   # raw stdout lines

def _parse_line(line):
    me, ms = RE_EPOCH.search(line), RE_STEP.search(line)
    if me and ms:
        epoch_records.append({
            'epoch': int(me.group(1)),
            'step':  int(ms.group(1)),
            's_gen':  float(RE_S_GEN.search(line).group(1))  if RE_S_GEN.search(line)  else None,
            's_disc': float(RE_S_DISC.search(line).group(1)) if RE_S_DISC.search(line) else None,
            'best':   float(RE_BEST.search(line).group(1))   if RE_BEST.search(line)   else None,
        })

# ── TensorBoard reader ────────────────────────────────────────────────────────

def _read_tb():
    """Load all scalar tags from the TensorBoard event file in LOG_DIR."""
    try:
        from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
        ea = EventAccumulator(LOG_DIR, size_guidance={'scalars': 0})
        ea.Reload()
        tags = ea.Tags().get('scalars', [])
        return {t: [(e.step, e.value) for e in ea.Scalars(t)] for t in tags}
    except Exception:
        return {}

# ── plot ──────────────────────────────────────────────────────────────────────

def _smooth(ys, window):
    if len(ys) < window:
        return ys
    return list(np.convolve(ys, np.ones(window) / window, mode='valid'))

def draw_live(fig, axes):
    tb = _read_tb()
    cur = epoch_records[-1]['epoch'] if epoch_records else 0
    ax1, ax2, ax3, ax4 = axes
    for ax in axes:
        ax.clear()

    # ── Panel 1: Gen & Disc smoothed loss (from stdout — real-time) ──────────
    ax1.set_title(f'Gen & Disc Loss  [epoch {cur}/{EPOCHS}]', fontweight='bold')
    sg = [(r['epoch'], r['s_gen'])  for r in epoch_records if r['s_gen']  is not None]
    sd = [(r['epoch'], r['s_disc']) for r in epoch_records if r['s_disc'] is not None]
    if sg:
        xs, ys = zip(*sg)
        ax1.plot(xs, ys, color='#4f46e5', lw=2, label='Gen (smoothed)')
        ax1.annotate(f'{ys[-1]:.4f}', xy=(xs[-1], ys[-1]), fontsize=8,
                     color='#4f46e5', xytext=(4, 4), textcoords='offset points')
    if sd:
        xs, ys = zip(*sd)
        ax1.plot(xs, ys, color='#dc2626', lw=2, label='Disc (smoothed)')
        ax1.annotate(f'{ys[-1]:.4f}', xy=(xs[-1], ys[-1]), fontsize=8,
                     color='#dc2626', xytext=(4, -10), textcoords='offset points')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.2)

    # ── Panel 2: Mel loss — perceptual quality proxy (from TensorBoard) ──────
    ax2.set_title('Mel Loss  (↓ = better voice quality)', fontweight='bold')
    mel = tb.get('loss/g/mel', [])
    if mel:
        xs, ys = zip(*mel)
        ax2.plot(xs, ys, color='#0ea5e9', lw=1, alpha=0.35, label='Raw')
        w = min(15, max(1, len(ys) // 5))
        sm = _smooth(ys, w)
        ax2.plot(xs[w - 1:], sm, color='#0369a1', lw=2, label=f'Smoothed (w={w})')
        ax2.annotate(f'{sm[-1]:.4f}', xy=(xs[-1], sm[-1]), fontsize=8, color='#0369a1',
                     xytext=(4, 4), textcoords='offset points')
    ax2.set_xlabel('Global Step'); ax2.set_ylabel('Mel L1 Loss')
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.2)

    # ── Panel 3: Feature-match, KL, adversarial (from TensorBoard) ───────────
    ax3.set_title('Sub-losses', fontweight='bold')
    sub = {
        'loss/g/fm':  ('Feat-match', '#10b981'),
        'loss/g/kl':  ('KL',         '#f59e0b'),
        'loss/g/adv': ('Adversarial', '#8b5cf6'),
    }
    for tag, (label, color) in sub.items():
        vals = tb.get(tag, [])
        if vals:
            xs, ys = zip(*vals)
            ax3.plot(xs, ys, color=color, lw=1.6, label=label, alpha=0.9)
            ax3.annotate(f'{ys[-1]:.4f}', xy=(xs[-1], ys[-1]), fontsize=7,
                         color=color, xytext=(4, 0), textcoords='offset points')
    ax3.set_xlabel('Global Step'); ax3.set_ylabel('Loss')
    ax3.legend(fontsize=9); ax3.grid(True, alpha=0.2)

    # ── Panel 4: Learning rate (from TensorBoard) ─────────────────────────────
    ax4.set_title('Learning Rate', fontweight='bold')
    lr = tb.get('learning_rate', [])
    if lr:
        xs, ys = zip(*lr)
        ax4.plot(xs, ys, color='#16a34a', lw=1.8)
        try:
            ax4.set_yscale('log')
        except Exception:
            pass
        ax4.annotate(f'{ys[-1]:.2e}', xy=(xs[-1], ys[-1]), fontsize=8,
                     color='#16a34a', xytext=(4, 4), textcoords='offset points')
    ax4.set_xlabel('Global Step'); ax4.set_ylabel('LR (log scale)')
    ax4.grid(True, alpha=0.2)

    fig.suptitle(f'{MODEL_NAME} — training monitor', fontsize=11, fontweight='bold')
    fig.tight_layout()

# ── launch training ───────────────────────────────────────────────────────────

print(f'Training {MODEL_NAME} for {EPOCHS} epochs on ' +
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Live plot refreshes every {REFRESH_SECS}s. Interrupt kernel to stop early.\n')

proc = subprocess.Popen([
    VENV_PY, 'core.py', 'train',
    '--model_name',             MODEL_NAME,
    '--sample_rate',            str(SAMPLE_RATE),
    '--total_epoch',            str(EPOCHS),
    '--save_every_epoch',       '10',
    '--batch_size',             '8',
    '--gpu',                    '0',
    '--vocoder',                'HiFi-GAN',
    '--overtraining_detector',  'True',
    '--overtraining_threshold', '50',
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
   text=True, bufsize=1, cwd=APPLIO_DIR)

def _stdout_reader():
    for line in proc.stdout:
        log_lines.append(line)
        _parse_line(line)

threading.Thread(target=_stdout_reader, daemon=True).start()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

try:
    while proc.poll() is None:
        clear_output(wait=True)
        draw_live(fig, axes)
        plt.show()
        for line in log_lines[-5:]:
            print(line, end='')
        time.sleep(REFRESH_SECS)
except KeyboardInterrupt:
    proc.terminate()
    print('\nTraining interrupted by user.')

proc.wait()
training_log_lines = log_lines

# Final draw after training ends
clear_output(wait=True)
draw_live(fig, axes)
plt.show()

for line in log_lines[-10:]:
    print(line, end='')

if proc.returncode not in (0, -15):   # 0 = done, -15 = SIGTERM (interrupted)
    raise RuntimeError(f'Training failed (exit {proc.returncode}) — see log above')

done_epoch = epoch_records[-1]['epoch'] if epoch_records else '?'
print(f'\n✅ Training complete — {done_epoch} epochs.')


In [ ]:
# ── Cell 6: Save model ─────────────────────────────────────────────────────────
import os, re, glob, shutil

log_dir      = f'{APPLIO_DIR}/logs/{MODEL_NAME}'
models_out   = f'{BASE_DIR}/models'
os.makedirs(models_out, exist_ok=True)

# Prefer the final named .pth; fall back to highest-epoch checkpoint
model_src = f'{log_dir}/{MODEL_NAME}.pth'
if not os.path.exists(model_src):
    ckpts = sorted(
        glob.glob(f'{log_dir}/*e_*.pth'),
        key=lambda fp: int(re.search(r'_(\d+)e_', fp).group(1))
                  if re.search(r'_(\d+)e_', fp) else 0
    )
    assert ckpts, f'No .pth found in {log_dir} — did training finish?'
    model_src = ckpts[-1]
    print(f'Using checkpoint: {os.path.basename(model_src)}')

index_files = glob.glob(f'{log_dir}/*.index')
assert index_files, f'No .index found in {log_dir}'

shutil.copy(model_src,      f'{models_out}/user_voice.pth')
shutil.copy(index_files[0], f'{models_out}/user_voice.index')

print(f'✓ Model → {models_out}/user_voice.pth  ({os.path.getsize(model_src)//1024//1024} MB)')
print(f'✓ Index → {models_out}/user_voice.index')
print('\n🎉 Voice training complete! Run Server.ipynb to start inference.')


In [ ]:
# ── Cell 7: Training diagnostics — loss curves, checkpoints, spectrogram ───────
# Re-runnable at any time after training. Requires Cell 5 in same session
# (uses training_log_lines as fallback if tensorboard events are empty).
import glob, os, re
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec

LOG_DIR = f'{APPLIO_DIR}/logs/{MODEL_NAME}'

gen_vals   = []
disc_vals  = []
extra_vals = {}

def load_scalars_from_dir(event_dir):
    try:
        from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
        ea = EventAccumulator(event_dir, size_guidance={'scalars': 0})
        ea.Reload()
        tags = ea.Tags().get('scalars', [])
        return ea, tags
    except Exception as e:
        print(f'  EventAccumulator error for {event_dir}: {e}')
        return None, []

candidate_dirs = {LOG_DIR}
for ef in glob.glob(f'{LOG_DIR}/**/events.out.tfevents.*', recursive=True):
    candidate_dirs.add(os.path.dirname(ef))

all_tags_found = []
for cdir in sorted(candidate_dirs):
    ea, tags = load_scalars_from_dir(cdir)
    if tags:
        all_tags_found.extend(tags)
        print(f'Tensorboard tags in {os.path.basename(cdir)}: {tags}')
        if not gen_vals:
            for tag in ('loss/g/total', 'loss_gen', 'train/loss_g', 'Generator', 'g/total'):
                if tag in tags:
                    gen_vals = [(e.step, e.value) for e in ea.Scalars(tag)]
                    print(f'  → Gen loss from "{tag}" ({len(gen_vals)} pts)')
                    break
        if not disc_vals:
            for tag in ('loss/d/total', 'loss_disc', 'train/loss_d', 'Discriminator', 'd/total'):
                if tag in tags:
                    disc_vals = [(e.step, e.value) for e in ea.Scalars(tag)]
                    print(f'  → Disc loss from "{tag}" ({len(disc_vals)} pts)')
                    break
        for tag in tags:
            tl = tag.lower()
            if any(k in tl for k in ('mel', 'fm', 'feat', 'kl')) and tag not in extra_vals:
                extra_vals[tag] = [(e.step, e.value) for e in ea.Scalars(tag)]

if not all_tags_found:
    print(f'No tensorboard events found under {LOG_DIR}')

# Fallback: parse stdout captured during training
if not gen_vals and not disc_vals:
    log_src = training_log_lines if 'training_log_lines' in dir() else []
    if log_src:
        print(f'Falling back to stdout parse ({len(log_src)} lines) ...')
        for line in log_src:
            mg = re.search(r'(?:loss[_/\s]?g(?:en)?|g_loss|Generator)\D+([\d.]+)', line, re.I)
            md = re.search(r'(?:loss[_/\s]?d(?:isc)?|d_loss|Discriminator)\D+([\d.]+)', line, re.I)
            me = re.search(r'[Ee]poch[\s:/\[]*([\d]+)', line)
            if me:
                ep = int(me.group(1))
                if mg: gen_vals.append((ep,  float(mg.group(1))))
                if md: disc_vals.append((ep, float(md.group(1))))
        if gen_vals: print(f'  → Parsed {len(gen_vals)} gen-loss points')

# Checkpoint sizes
ckpt_files = sorted(
    glob.glob(f'{LOG_DIR}/**/*e_*.pth', recursive=True) + glob.glob(f'{LOG_DIR}/*e_*.pth'),
    key=lambda f: int(re.search(r'_(\d+)e_', f).group(1)) if re.search(r'_(\d+)e_', f) else 0
)
seen = set(); ckpt_files = [f for f in ckpt_files if not (f in seen or seen.add(f))]
ckpt_epochs = []; ckpt_sizes = []
for fp in ckpt_files:
    m = re.search(r'_(\d+)e_', os.path.basename(fp))
    if m:
        ckpt_epochs.append(int(m.group(1)))
        ckpt_sizes.append(os.path.getsize(fp) / 1024 / 1024)

training_wav = f'{BASE_DIR}/dataset/{MODEL_NAME}/voice_recording.wav'
has_spectrogram = os.path.exists(training_wav)
has_loss  = bool(gen_vals or disc_vals)
has_extra = bool(extra_vals)
has_ckpts = bool(ckpt_epochs)
n_rows = int(has_loss) + int(has_extra) + int(has_ckpts) + int(has_spectrogram)

if n_rows == 0:
    print('\nNothing to plot yet.')
else:
    fig = plt.figure(figsize=(13, 4.2 * n_rows))
    gs  = GridSpec(n_rows, 1, figure=fig, hspace=0.5)
    row = 0

    if has_loss:
        ax = fig.add_subplot(gs[row]); row += 1
        colors = {'gen': '#4f46e5', 'disc': '#dc2626'}
        if gen_vals:
            xs, ys = zip(*gen_vals)
            ax.plot(xs, ys, label='Generator loss', color=colors['gen'], linewidth=1.6)
            ax.annotate(f'start {ys[0]:.3f}',  xy=(xs[0],  ys[0]),  fontsize=7.5, color=colors['gen'])
            ax.annotate(f'end {ys[-1]:.3f}',   xy=(xs[-1], ys[-1]), fontsize=7.5, color=colors['gen'], ha='right')
        if disc_vals:
            xs, ys = zip(*disc_vals)
            ax.plot(xs, ys, label='Discriminator loss', color=colors['disc'], linewidth=1.6, alpha=0.85)
            ax.annotate(f'start {ys[0]:.3f}',  xy=(xs[0],  ys[0]),  fontsize=7.5, color=colors['disc'])
            ax.annotate(f'end {ys[-1]:.3f}',   xy=(xs[-1], ys[-1]), fontsize=7.5, color=colors['disc'], ha='right')
        ax.set_title('Generator & Discriminator Loss', fontsize=12, fontweight='bold')
        ax.set_xlabel('Global Step'); ax.set_ylabel('Loss')
        ax.legend(framealpha=0.9); ax.grid(True, alpha=0.25)
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))

    if has_extra:
        ax2 = fig.add_subplot(gs[row]); row += 1
        palette = ['#0ea5e9', '#10b981', '#f59e0b', '#8b5cf6', '#ef4444']
        for (tag, vals), col in zip(extra_vals.items(), palette):
            if vals:
                xs, ys = zip(*vals)
                ax2.plot(xs, ys, label=tag.split('/')[-1], color=col, linewidth=1.4, alpha=0.85)
        ax2.set_title('Sub-losses (mel / feature-match / KL)', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Global Step'); ax2.set_ylabel('Loss')
        ax2.legend(framealpha=0.9, ncol=3, fontsize=9); ax2.grid(True, alpha=0.25)

    if has_ckpts:
        ax3 = fig.add_subplot(gs[row]); row += 1
        bar_w = max(2, (max(ckpt_epochs)-min(ckpt_epochs))/len(ckpt_epochs)*0.6) if len(ckpt_epochs)>1 else 5
        ax3.bar(ckpt_epochs, ckpt_sizes, color='#6366f1', alpha=0.85, width=bar_w)
        ax3.set_title(f'Checkpoint Sizes ({len(ckpt_epochs)} saved)', fontsize=12, fontweight='bold')
        ax3.set_xlabel('Epoch'); ax3.set_ylabel('Size (MB)')
        ax3.grid(True, alpha=0.2, axis='y')
        for ep, sz in zip(ckpt_epochs, ckpt_sizes):
            ax3.text(ep, sz+max(ckpt_sizes)*0.012, f'{sz:.0f}', ha='center', va='bottom', fontsize=8)

    if has_spectrogram:
        try:
            import scipy.io.wavfile as wf, scipy.signal as sig
            sr, data = wf.read(training_wav)
            if data.ndim > 1: data = data[:, 0]
            dtype_max = np.iinfo(data.dtype).max if np.issubdtype(data.dtype, np.integer) else 1.0
            data = data[:sr*30].astype(np.float32) / dtype_max
            f, t, Sxx = sig.spectrogram(data, sr, nperseg=1024, noverlap=768, window='hann')
            ax4 = fig.add_subplot(gs[row]); row += 1
            freq_limit = min(len(f), int(len(f)*8000/(sr/2)))
            img = ax4.pcolormesh(t, f[:freq_limit]/1000,
                                 10*np.log10(Sxx[:freq_limit]+1e-10),
                                 shading='gouraud', cmap='magma', vmin=-80, vmax=0)
            plt.colorbar(img, ax=ax4, label='dB', pad=0.01, fraction=0.02)
            ax4.set_title('Training Voice — Input Spectrogram (first 30 s)', fontsize=12, fontweight='bold')
            ax4.set_xlabel('Time (s)'); ax4.set_ylabel('Frequency (kHz)')
        except Exception as spec_err:
            print(f'  Spectrogram skipped: {spec_err}')

    plt.suptitle(f'Model: {MODEL_NAME}  |  {EPOCHS} epochs target', fontsize=13, fontweight='bold', y=1.01)
    plot_path = f'{BASE_DIR}/training_plots.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n✓ Plot saved → {plot_path}')
    if has_loss:
        if gen_vals:  print(f'  Gen loss:  {gen_vals[0][1]:.4f} → {gen_vals[-1][1]:.4f}  ({"↓ converging" if gen_vals[-1][1] < gen_vals[0][1] else "↑ check training"})')
        if disc_vals: print(f'  Disc loss: {disc_vals[0][1]:.4f} → {disc_vals[-1][1]:.4f}')
